# PROYECTO PRODUCTIVO IB · SEMANA 8
## Organización y almacenamiento de datos
**Caso: DATAPREP 360**

### Entrada
`dataprep360_validado.csv` generado en Semana 7.

### Producto
Base o repositorio de datos organizado y documentado.

### Construiremos
- estructura de carpetas;
- tablas y relaciones;
- base SQLite;
- diccionario de datos;
- manifiesto de versión;
- exportación controlada;
- respaldo;
- prueba de recuperación.

## BLOQUE 1 · Librerías

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import shutil
import json
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

print("Entorno preparado.")

## BLOQUE 2 · Cargar el producto de Semana 7

In [ ]:
def cargar_validado():
    try:
        from google.colab import files
        print("Seleccione dataprep360_validado.csv generado en Semana 7.")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No se seleccionó ningún archivo.")
        ruta = Path(next(iter(uploaded)))
    except ImportError:
        ruta = Path(input("Ruta de dataprep360_validado.csv: ").strip())

    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")

    for enc in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            return pd.read_csv(ruta, encoding=enc), ruta
        except UnicodeDecodeError:
            pass

    raise UnicodeError("No fue posible leer el CSV con las codificaciones previstas.")

df, RUTA_ORIGINAL = cargar_validado()

print("Archivo:", RUTA_ORIGINAL.name)
print("Dimensiones:", df.shape)
display(df.head())

## BLOQUE 3 · Verificar columnas núcleo

In [ ]:
COLUMNAS_NUCLEO = [
    "registro_id", "fecha_evento", "sede", "categoria", "canal",
    "cantidad", "valor_soles", "duracion_min", "estado",
    "codigo_referencia", "fuente", "observacion"
]

faltantes = [c for c in COLUMNAS_NUCLEO if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas núcleo: {faltantes}")

print("OK: estructura mínima disponible.")

## BLOQUE 4 · Crear repositorio

In [ ]:
RAIZ = Path("/content/DATAPREP360")

# Si se vuelve a ejecutar todo el notebook, recreamos el laboratorio de forma controlada.
if RAIZ.exists():
    shutil.rmtree(RAIZ)

CARPETAS = {
    "entrada": RAIZ / "01_entrada",
    "base": RAIZ / "02_base_datos",
    "documentacion": RAIZ / "03_documentacion",
    "exportaciones": RAIZ / "04_exportaciones",
    "respaldos": RAIZ / "05_respaldos"
}

for ruta in CARPETAS.values():
    ruta.mkdir(parents=True, exist_ok=True)

for nombre, ruta in CARPETAS.items():
    print(f"{nombre:15} -> {ruta}")

## BLOQUE 5 · Conservar entrada y establecer versión

In [ ]:
VERSION = "v1"
FECHA_VERSION = datetime.now().strftime("%Y-%m-%d")

RUTA_ENTRADA = CARPETAS["entrada"] / (
    f"dataprep360_validado_{VERSION}_{FECHA_VERSION}.csv"
)

# Guardamos una copia reproducible del dataframe cargado.
df.to_csv(RUTA_ENTRADA, index=False, encoding="utf-8-sig")

print("Entrada conservada:", RUTA_ENTRADA.name)

## BLOQUE 6 · Explorar cardinalidad de entidades de referencia

In [ ]:
for variable in ["sede", "categoria", "canal", "fuente"]:
    print(
        f"{variable:12} -> "
        f"{df[variable].nunique(dropna=True)} valores distintos"
    )

## BLOQUE 7 · Crear tablas de referencia

In [ ]:
def crear_dimension(dataframe, columna, nombre_id):
    dimension = (
        dataframe[[columna]]
        .dropna()
        .drop_duplicates()
        .sort_values(columna)
        .reset_index(drop=True)
    )
    dimension.insert(0, nombre_id, range(1, len(dimension) + 1))
    return dimension

dim_sede = crear_dimension(df, "sede", "sede_id")
dim_categoria = crear_dimension(df, "categoria", "categoria_id")
dim_canal = crear_dimension(df, "canal", "canal_id")
dim_fuente = crear_dimension(df, "fuente", "fuente_id")

display(dim_sede)
display(dim_categoria)
display(dim_canal)
display(dim_fuente)

## BLOQUE 8 · Construir la tabla EVENTO con claves foráneas

In [ ]:
evento = df[COLUMNAS_NUCLEO].copy()

evento = evento.merge(dim_sede, on="sede", how="left")
evento = evento.merge(dim_categoria, on="categoria", how="left")
evento = evento.merge(dim_canal, on="canal", how="left")
evento = evento.merge(dim_fuente, on="fuente", how="left")

display(
    evento[
        [
            "registro_id",
            "sede", "sede_id",
            "categoria", "categoria_id",
            "canal", "canal_id",
            "fuente", "fuente_id"
        ]
    ].head(10)
)

## BLOQUE 9 · Seleccionar estructura definitiva de EVENTO

In [ ]:
evento = evento[
    [
        "registro_id",
        "fecha_evento",
        "sede_id",
        "categoria_id",
        "canal_id",
        "fuente_id",
        "cantidad",
        "valor_soles",
        "duracion_min",
        "estado",
        "codigo_referencia",
        "observacion"
    ]
].copy()

# Convertir IDs a entero si no existen nulos.
for c in ["sede_id", "categoria_id", "canal_id", "fuente_id"]:
    if evento[c].isna().any():
        raise ValueError(f"Se generaron claves sin correspondencia en {c}.")
    evento[c] = evento[c].astype(int)

print("Entrada :", len(df))
print("EVENTO  :", len(evento))
assert len(df) == len(evento), "La transformación cambió el número de eventos."

display(evento.head())

## BLOQUE 10 · Crear base SQLite y esquema relacional

In [ ]:
RUTA_BD = CARPETAS["base"] / (
    f"dataprep360_bd_{VERSION}_{FECHA_VERSION}.sqlite"
)

conn = sqlite3.connect(RUTA_BD)
conn.execute("PRAGMA foreign_keys = ON;")

cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS evento;
DROP TABLE IF EXISTS dim_sede;
DROP TABLE IF EXISTS dim_categoria;
DROP TABLE IF EXISTS dim_canal;
DROP TABLE IF EXISTS dim_fuente;

CREATE TABLE dim_sede (
    sede_id INTEGER PRIMARY KEY,
    sede TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_categoria (
    categoria_id INTEGER PRIMARY KEY,
    categoria TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_canal (
    canal_id INTEGER PRIMARY KEY,
    canal TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_fuente (
    fuente_id INTEGER PRIMARY KEY,
    fuente TEXT NOT NULL UNIQUE
);

CREATE TABLE evento (
    registro_id TEXT PRIMARY KEY,
    fecha_evento TEXT NOT NULL,
    sede_id INTEGER NOT NULL,
    categoria_id INTEGER NOT NULL,
    canal_id INTEGER NOT NULL,
    fuente_id INTEGER NOT NULL,
    cantidad INTEGER,
    valor_soles REAL,
    duracion_min REAL,
    estado TEXT,
    codigo_referencia TEXT,
    observacion TEXT,
    FOREIGN KEY (sede_id) REFERENCES dim_sede(sede_id),
    FOREIGN KEY (categoria_id) REFERENCES dim_categoria(categoria_id),
    FOREIGN KEY (canal_id) REFERENCES dim_canal(canal_id),
    FOREIGN KEY (fuente_id) REFERENCES dim_fuente(fuente_id)
);
""")

conn.commit()
print("Esquema relacional creado.")

## BLOQUE 11 · Insertar tablas padre y tabla EVENTO

In [ ]:
# to_sql no reemplaza el esquema: inserta en las tablas ya creadas.
dim_sede.to_sql("dim_sede", conn, if_exists="append", index=False)
dim_categoria.to_sql("dim_categoria", conn, if_exists="append", index=False)
dim_canal.to_sql("dim_canal", conn, if_exists="append", index=False)
dim_fuente.to_sql("dim_fuente", conn, if_exists="append", index=False)
evento.to_sql("evento", conn, if_exists="append", index=False)

conn.commit()
print("Datos almacenados.")

## BLOQUE 12 · Verificar tablas y cantidad de registros

In [ ]:
tablas = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
    """,
    conn
)
display(tablas)

nombres_tablas = [
    "dim_sede", "dim_categoria", "dim_canal", "dim_fuente", "evento"
]

resumen = []
for tabla in nombres_tablas:
    n = pd.read_sql_query(
        f"SELECT COUNT(*) AS n FROM {tabla};", conn
    ).loc[0, "n"]
    resumen.append({"tabla": tabla, "registros": int(n)})

resumen_tablas = pd.DataFrame(resumen)
display(resumen_tablas)

## BLOQUE 13 · Comprobar integridad referencial

In [ ]:
integridad = pd.read_sql_query(
    "PRAGMA foreign_key_check;",
    conn
)

if integridad.empty:
    print("OK: no se detectaron violaciones de claves foráneas.")
else:
    print("ATENCIÓN: existen violaciones de integridad referencial.")
    display(integridad)

## BLOQUE 14 · Recuperar información mediante JOIN

In [ ]:
consulta = """
SELECT
    e.registro_id,
    e.fecha_evento,
    s.sede,
    c.categoria,
    ca.canal,
    f.fuente,
    e.cantidad,
    e.valor_soles,
    e.duracion_min,
    e.estado,
    e.codigo_referencia,
    e.observacion
FROM evento AS e
LEFT JOIN dim_sede AS s
    ON e.sede_id = s.sede_id
LEFT JOIN dim_categoria AS c
    ON e.categoria_id = c.categoria_id
LEFT JOIN dim_canal AS ca
    ON e.canal_id = ca.canal_id
LEFT JOIN dim_fuente AS f
    ON e.fuente_id = f.fuente_id
LIMIT 10;
"""

resultado = pd.read_sql_query(consulta, conn)
display(resultado)

## BLOQUE 15 · Crear exportación controlada

In [ ]:
consulta_completa = """
SELECT
    e.registro_id,
    e.fecha_evento,
    s.sede,
    c.categoria,
    ca.canal,
    f.fuente,
    e.cantidad,
    e.valor_soles,
    e.duracion_min,
    e.estado,
    e.codigo_referencia,
    e.observacion
FROM evento AS e
LEFT JOIN dim_sede AS s ON e.sede_id = s.sede_id
LEFT JOIN dim_categoria AS c ON e.categoria_id = c.categoria_id
LEFT JOIN dim_canal AS ca ON e.canal_id = ca.canal_id
LEFT JOIN dim_fuente AS f ON e.fuente_id = f.fuente_id;
"""

df_exportado = pd.read_sql_query(consulta_completa, conn)

RUTA_EXPORTACION = CARPETAS["exportaciones"] / (
    f"dataprep360_exportacion_{VERSION}_{FECHA_VERSION}.csv"
)

df_exportado.to_csv(
    RUTA_EXPORTACION,
    index=False,
    encoding="utf-8-sig"
)

print("Exportación:", RUTA_EXPORTACION.name)

## BLOQUE 16 · Generar diccionario de datos desde SQLite

In [ ]:
descripciones = {
    "registro_id": "Identificador único del evento",
    "fecha_evento": "Fecha asociada al evento",
    "sede_id": "Identificador de la sede",
    "categoria_id": "Identificador de la categoría",
    "canal_id": "Identificador del canal",
    "fuente_id": "Identificador de la fuente",
    "cantidad": "Cantidad registrada en el evento",
    "valor_soles": "Valor monetario expresado en soles",
    "duracion_min": "Duración del evento en minutos",
    "estado": "Estado asociado al registro",
    "codigo_referencia": "Código de referencia",
    "observacion": "Observación complementaria",
    "sede": "Nombre estandarizado de la sede",
    "categoria": "Nombre de la categoría",
    "canal": "Nombre del canal",
    "fuente": "Nombre de la fuente"
}

filas_diccionario = []

for tabla in nombres_tablas:
    info = pd.read_sql_query(f"PRAGMA table_info({tabla});", conn)

    for _, fila in info.iterrows():
        campo = fila["name"]
        rol = "PK" if fila["pk"] == 1 else "Atributo"
        referencia = ""

        if tabla == "evento" and campo in {
            "sede_id", "categoria_id", "canal_id", "fuente_id"
        }:
            rol = "FK"
            mapa_ref = {
                "sede_id": "dim_sede.sede_id",
                "categoria_id": "dim_categoria.categoria_id",
                "canal_id": "dim_canal.canal_id",
                "fuente_id": "dim_fuente.fuente_id"
            }
            referencia = mapa_ref[campo]

        filas_diccionario.append({
            "tabla": tabla,
            "campo": campo,
            "tipo": fila["type"],
            "rol": rol,
            "obligatorio": "Sí" if fila["notnull"] == 1 else "No",
            "descripcion": descripciones.get(campo, ""),
            "referencia": referencia
        })

diccionario = pd.DataFrame(filas_diccionario)
display(diccionario)

RUTA_DICCIONARIO = CARPETAS["documentacion"] / (
    f"diccionario_datos_{VERSION}_{FECHA_VERSION}.csv"
)

diccionario.to_csv(
    RUTA_DICCIONARIO,
    index=False,
    encoding="utf-8-sig"
)

## BLOQUE 17 · Generar manifiesto de versión

In [ ]:
manifiesto = {
    "proyecto": "DATAPREP 360",
    "semana": 8,
    "actividad": "Organización y almacenamiento",
    "version": VERSION,
    "fecha_generacion": datetime.now().isoformat(timespec="seconds"),
    "archivo_fuente": RUTA_ENTRADA.name,
    "base_datos": RUTA_BD.name,
    "motor": "SQLite",
    "tablas": nombres_tablas,
    "numero_eventos": int(len(evento)),
    "diccionario_datos": RUTA_DICCIONARIO.name,
    "exportacion": RUTA_EXPORTACION.name
}

RUTA_MANIFIESTO = CARPETAS["documentacion"] / (
    f"manifiesto_{VERSION}_{FECHA_VERSION}.json"
)

with open(RUTA_MANIFIESTO, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, ensure_ascii=False, indent=4)

print(json.dumps(manifiesto, ensure_ascii=False, indent=4))

## BLOQUE 18 · Cerrar base y mostrar repositorio

In [ ]:
conn.commit()
conn.close()

for raiz, carpetas, archivos in os.walk(RAIZ):
    nivel = raiz.replace(str(RAIZ), "").count(os.sep)
    sangria = "    " * nivel
    print(f"{sangria}{Path(raiz).name}/")
    for archivo in archivos:
        print(f"{sangria}    {archivo}")

## BLOQUE 19 · Crear respaldo

El ZIP temporal se crea fuera del repositorio para evitar que el respaldo intente incluirse a sí mismo.

In [ ]:
RUTA_ZIP_TEMP = Path("/content") / (
    f"DATAPREP360_respaldo_{VERSION}_{FECHA_VERSION}"
)

archivo_zip = shutil.make_archive(
    str(RUTA_ZIP_TEMP),
    "zip",
    root_dir=RAIZ
)

DESTINO_RESPALDO = CARPETAS["respaldos"] / (
    f"DATAPREP360_respaldo_{VERSION}_{FECHA_VERSION}.zip"
)

shutil.move(archivo_zip, DESTINO_RESPALDO)

print("Respaldo:", DESTINO_RESPALDO)
print("Existe:", DESTINO_RESPALDO.exists())
print("Tamaño KB:", round(DESTINO_RESPALDO.stat().st_size / 1024, 2))

## BLOQUE 20 · Simular recuperación del respaldo

In [ ]:
RUTA_RECUPERACION = Path("/content/DATAPREP360_RECUPERADO")

if RUTA_RECUPERACION.exists():
    shutil.rmtree(RUTA_RECUPERACION)

RUTA_RECUPERACION.mkdir(parents=True)

shutil.unpack_archive(
    str(DESTINO_RESPALDO),
    str(RUTA_RECUPERACION)
)

bases_recuperadas = list(RUTA_RECUPERACION.rglob("*.sqlite"))

if not bases_recuperadas:
    raise FileNotFoundError(
        "El respaldo se extrajo, pero no contiene una base SQLite."
    )

BD_RECUPERADA = bases_recuperadas[0]
print("Base recuperada:", BD_RECUPERADA)

## BLOQUE 21 · Verificar que la base recuperada funciona

In [ ]:
conn_rec = sqlite3.connect(BD_RECUPERADA)
conn_rec.execute("PRAGMA foreign_keys = ON;")

conteo = pd.read_sql_query(
    "SELECT COUNT(*) AS registros FROM evento;",
    conn_rec
)

n_original = len(evento)
n_recuperado = int(conteo.loc[0, "registros"])

integridad_rec = pd.read_sql_query(
    "PRAGMA foreign_key_check;",
    conn_rec
)

conn_rec.close()

print("Eventos originales  :", n_original)
print("Eventos recuperados :", n_recuperado)
print(
    "Integridad FK       :",
    "OK" if integridad_rec.empty else "REVISAR"
)

if n_original == n_recuperado and integridad_rec.empty:
    print("\nRECUPERACIÓN VERIFICADA.")
else:
    print("\nREVISAR RESPALDO O BASE RECUPERADA.")

## BLOQUE 22 · Reporte final

In [ ]:
estado_recuperacion = (
    "VERIFICADA"
    if n_original == n_recuperado and integridad_rec.empty
    else "REVISAR"
)

print("=" * 65)
print("DATAPREP 360 · SEMANA 8")
print("ORGANIZACIÓN Y ALMACENAMIENTO")
print("=" * 65)

print(f"""
Archivo de entrada:
{RUTA_ENTRADA.name}

Eventos almacenados:
{len(evento)}

Tablas:
{len(nombres_tablas)}

Base de datos:
{RUTA_BD.name}

Diccionario:
{RUTA_DICCIONARIO.name}

Manifiesto:
{RUTA_MANIFIESTO.name}

Exportación:
{RUTA_EXPORTACION.name}

Respaldo:
{DESTINO_RESPALDO.name}

Recuperación:
{estado_recuperacion}
""")

## Producto final de Semana 8

El repositorio contiene:
- entrada controlada;
- base SQLite;
- tablas relacionadas;
- diccionario de datos;
- manifiesto;
- exportación;
- respaldo;
- evidencia de recuperación.

**Resultado:** base o repositorio de datos organizado y documentado.